In [1]:
%load_ext autoreload
%autoreload 2

In [64]:
import numpy as np
import pandas as pd
import xarray as xr
import dataclasses
import natsort

import tensorflow as tf
tf.random.set_seed(42)
from tensorflow_probability import distributions as tfd

from meridian import constants as c
from meridian.mini.mini_utils import InputDataBuilderMini, \
  MediaTensorsMini, KPITensorsMini, RFTensorsMini, ControlsTensorsMini


In [22]:
# Load the data
df = pd.read_csv("./../data/simulated_data/csv/geo_media_rf.csv")

# Define the config
model_config = {

  # time and geo inputs
  'time_col': 'time',
  'geo_col': 'geo',  # assumed to be national model if not given
  'population_col': 'population', # mandatory if geo_col is given

  # kpi inputs
  'kpi_col': 'conversions',  #
  'kpi_type': 'non_revenue',
  'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],

  # reach based media inputs
  'reach_cols': ['Channel3_reach'],
  'frequency_cols': ['Channel3_frequency'],
  'rf_spend_cols': ['Channel3_spend'],
  'rf_channels': ['Channel3'],

  # control inputs
  'control_cols': ["sentiment_score_control", "competitor_activity_score_control"]

  }


In [25]:
input_data = InputDataBuilderMini(model_config).build(df)
media_tensors = MediaTensorsMini.build(input_data)
rf_tensors = RFTensorsMini.build(input_data)
controls_tensors = ControlsTensorsMini.build(input_data)
kpi_tensors = KPITensorsMini.build(input_data)

In [90]:
tf.random.set_seed(42)

# time varying intercept
n_knots=10
knot_info = input_data.knot_info(n_knots)
knot_weights = tf.convert_to_tensor(knot_info.weights)

knot_values = tfd.BatchBroadcast(
  tfd.Normal(0.0, 5.0),
  n_knots,
  name=c.KNOT_VALUES
)

mu_t = tf.einsum("k, kt -> t", knot_values.sample(), knot_weights)


In [94]:
# geo level intercept
id(input_data.controls)

6070131232

In [ ]:
  # tau_g_excl_baseline: backend.tfd.Distribution = dataclasses.field(
  #     default_factory=lambda: backend.tfd.Normal(
  #         0.0, 5.0, name=constants.TAU_G_EXCL_BASELINE
  #     ),
  # )

In [95]:
num1 = 11
print(id(num1))
num1 = 12
print(id(num1))


4336166064
4336166096


In [96]:
id(11)

4336166064

In [98]:
dict1 = {'value': 11}
dict2 = dict1

print(id(dict1))
print(id(dict2))

dict2['value'] = 12
print(id(dict2))


6102696064
6102696064
6102696064


In [99]:
dict1

{'value': 12}

In [107]:
l1 = ['x', 'y', 'z']
print(id(l1))
print(id(l1[0]))
print(id(l1[1]))
print(id(l1[2]))

6102406464
4336229448
4336229496
4336229544


In [4]:
i = 'a'
print(id(i))

import sys
sys.getsizeof(i)

4361345016


42

In [3]:
28/8

3.5